# Fine-tune the Kurrent adapter on Kade lines, right here in Jupyter

This notebook trains on **this notebook instance's own GPU**: no training job, the output appears
below the cells. It runs the same script as `sagemaker_kade_lora.ipynb`
(`sagemaker/code/finetune_qwen_kurrent_lora.py`), so results from the two are comparable.
`docs/lora-finetuning-explained.md` explains every step in plain language.

**The notebook instance must have a GPU.** When you create it (SageMaker → Notebook instances →
Create):

| Setting | Value |
|---|---|
| Notebook instance type | `ml.g4dn.xlarge` (T4, 16 GB, about $0.74/h) or `ml.g5.xlarge` (A10G, 24 GB, about $1.41/h) |
| Volume size | **50 GB** (the default 5 GB can't hold the 9 GB model) |
| Kernel, once it's running | `conda_pytorch` (any `conda_pytorch_*`) |

**A GPU notebook instance bills every hour it is running, even with the browser closed.** Stop it
(SageMaker → Notebook instances → Stop) as soon as you're done. Forgotten over a weekend, a g5 is
about $70.

Run the cells from top to bottom. Start with `RUN = "smoke"` in step 4.

## 1. Install the libraries

Qwen3-VL needs `transformers` 4.57 or newer. Packages you install are wiped every time the instance is stopped, so run this cell again after each start. Run it **before** anything imports `transformers`; if you already did, restart the kernel (Kernel → Restart) and start here.

In [ ]:
%pip install -q "transformers==4.57.6" "peft==0.21.2" "accelerate>=1.10"

## 2. Check the GPU and the disk

In [ ]:
import os
import shutil
import torch

WORK = "/home/ec2-user/SageMaker/kade-lora"   # on the instance's volume: survives a Stop, not a Delete
os.makedirs(WORK, exist_ok=True)
# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Stop this instance and change its type to ml.g4dn.xlarge or ml.g5.xlarge.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, "
      f"{'bf16' if torch.cuda.get_device_capability()[0] >= 8 else 'fp16 (no bf16 on this GPU)'}")
free = shutil.disk_usage(WORK).free / 1e9
print(f"Disk: {free:.0f} GB free in {WORK}")
if free < 20:
    print("WARNING: under 20 GB free. Stop the instance and raise its volume size to 50 GB.")

## 3. Get the data

You need `kade_lines.zip`: the line crops made on Ubuntu with `notebooks/kade_line_segmentation.ipynb`
(`docs/ubuntu-setup.md`), zipped with
`cd ~/badger-scribe-data && python -m zipfile -c kade_lines.zip kade_lines/`. Drag it into this
folder in the Jupyter file browser on the left.

`train.csv` comes from the class's shared bucket. If you can't reach it, drag `train.csv` in too.

This pairs each crop with its line of the label, for pages where kraken found as many lines as the
label has (the `Postmark` line doesn't count; it isn't in the image). Two documents are kept out of
training and used for scoring instead: `kade_112`, the team's dev set, and `kade_007`, a full-size
document held out so we can see how the model does on letters it has never seen. That is exactly
the situation on Kaggle, where every test page comes from a document that isn't in the training
data. The Kade test pages that were cut into lines are listed as well, so the model can read them
at the end.

**Check the printed lines:** if well under 70% of training pages are usable, look at the
segmentation before training. If it finds fewer than 141 test pages, the segmentation notebook
hasn't cut all of them yet.

In [ ]:
import sys

repo_dir = os.path.abspath("..")   # this notebook lives in <repo>/notebooks
script = os.path.join(repo_dir, "sagemaker", "code", "finetune_qwen_kurrent_lora.py")
sys.path.insert(0, os.path.join(repo_dir, "sagemaker"))
import prepare_kade_lines

lines_zip = "kade_lines.zip"       # where you dragged it
train_csv = os.path.join(WORK, "train.csv")
if os.path.exists("train.csv"):
    shutil.copy("train.csv", train_csv)
elif not os.path.exists(train_csv):
    import boto3
    boto3.client("s3").download_file("ml-marathon-shared", "badger-scribe/train.csv", train_csv)

data_dir = os.path.join(WORK, "data")
stats = prepare_kade_lines.build(lines_zip, train_csv, data_dir)

## 4. Choose the run and the settings

Do the runs roughly in this order, because each one answers a question the next one depends on.

| Run | What it does | The question it answers | About how long on g5 |
|---|---|---|---|
| `smoke` | 64 lines, 1 epoch, 2 pages scored | Does everything work? | 10–15 min (mostly the model download, once) |
| `baseline` | No training. Scores the held-out pages and reads the test pages with the Kurrent adapter as it is | What number does training have to beat, on our pages and on the leaderboard? | 20–30 min |
| `full` | The real fine-tune: all lines, scored after every epoch | Does fine-tuning help, and by how much? | 45–60 min |
| `repeat` | `full` again with a different random seed | How much does the score move by chance alone? | 45–60 min |
| `no_kurrent` | `full`, but starting from plain Qwen3-VL without the Bern adapter | Does the Bern adapter matter, or would fine-tuning alone get us there? | 45–60 min |
| `final` | `full` trained on `kade_007` too, keeping the last epoch, then reads the test pages | This is the model we submit | 60–75 min |

On g4dn, double the times. Before the `final` run, set `epochs` to the best epoch of your best
`full` run. Once `kade_007` is part of training there is nothing large left to choose the best
epoch with, so we decide it in advance.

The settings are explained in `docs/lora-finetuning-explained.md`, under "The settings you can
change". **Change one at a time.** Each combination gets its own folder, and running the same
combination again **continues** it from its last saved epoch, which is useful after a crash. To
start it over, delete its folder under `kade-lora/runs/`.

In [ ]:
RUN = "smoke"   # smoke -> baseline -> full -> repeat -> no_kurrent -> final

settings = {
    "lr": 1e-4,           # learning rate: size of each nudge
    "epochs": 3,          # passes over all training lines
    "lora_r": 16,         # adapter size
    "lora_alpha": 32,     # adapter strength; keep at 2 x lora_r
    "lora_dropout": 0.05,
    "batch_size": 4,      # lines per step; halve it (and double grad_accum) if the GPU runs out of memory
    "grad_accum": 4,      # effective batch = batch_size x grad_accum = 16
    "min_pixels": 65536,  # smallest image size the model sees; higher = more detail, slower
    "seed": 42,
}
if RUN == "smoke":
    settings.update(epochs=1, max_train_lines=64, dev_pages=2)
elif RUN == "baseline":
    settings = {"epochs": 0, "predict_test": 1}
elif RUN == "repeat":
    settings.update(seed=7)
elif RUN == "no_kurrent":
    settings.update(start_adapter="none")
elif RUN == "final":
    settings.update(holdout_docs="none", keep="last", predict_test=1)   # and set epochs to your best full run's best epoch

run_name = RUN if RUN in ("smoke", "baseline") else RUN + "-" + "-".join(f"{k}{v}" for k, v in settings.items())
run_dir = os.path.join(WORK, "runs", run_name)
for sub in ("model", "checkpoints", "output"):
    os.makedirs(os.path.join(run_dir, sub), exist_ok=True)
print(run_dir)

## 5. Train

The log streams below and is also saved to `train.log` in the run folder. If the browser
disconnects, the training keeps going as long as the instance is running; reopen the notebook and
run `!tail -n 30 {log}` to see where it got to. To stop it, use Kernel → Interrupt.

Lines to look for: `[dev epoch0] ... mean page CER` (the score before training, with each held-out
document listed separately), `loss` going down, and a `[dev epochN]` line after each epoch.

In [ ]:
args = {**settings,
        "train_dir": data_dir,
        "model_dir": os.path.join(run_dir, "model"),
        "checkpoint_dir": os.path.join(run_dir, "checkpoints"),
        "output_data_dir": os.path.join(run_dir, "output")}
flags = " ".join(f"--{k} {v}" for k, v in args.items())
log = os.path.join(run_dir, "train.log")
python = sys.executable   # this kernel's own Python, where step 1 installed the libraries
!{python} {script} {flags} 2>&1 | tee -a {log}

## 6. Results

Epoch 0 is the Kurrent adapter before any training. Every document gets two scores:

- **CER** is scored exactly the way Kaggle scores it. This is the number to compare with other teams.
- **Reading CER** leaves out the label's `Postmark` line. The postmark is on the other side of the
  card, so no model can read it, and this number shows how well the model reads what is actually
  in the picture.

The report also gives a **95% interval** for the drop in CER. If the whole interval is above 0, the
improvement is real and not just luck in which pages happened to be scored.

In [ ]:
import json
import pandas as pd
import kade_report

out = os.path.join(run_dir, "model")
result = kade_report.report(out)

log_json = json.load(open(os.path.join(out, "train_log.json")))
curve = pd.DataFrame(log_json["dev_cer_by_epoch"])
if log_json["loss"]:
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
    loss = pd.DataFrame(log_json["loss"])
    a.plot(loss.step, loss.loss); a.set_xlabel("step"); a.set_ylabel("training loss")
    b.plot(curve.epoch, curve.dev_cer, marker="o", label="CER")
    b.plot(curve.epoch, curve.reading_cer, marker="o", label="reading CER")
    b.set_xlabel("epoch"); b.set_ylabel("held-out CER (lower is better)"); b.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Paste this line into the table in docs/kade-lora-results.md, so the whole team sees every run.
print(kade_report.results_row(out, run_name))

## 7. Make a Kaggle submission (`baseline` and `final` runs)

These two runs also read the 141 Kade test pages, and this cell puts those transcriptions into a
full submission file.

`base_csv` is the submission we start from. If the team has a full submission that covers all
three categories (for example a CHURRO run), use it, and only its Kade rows are replaced. If there
isn't one yet, use Kaggle's `sample_submission.csv`. Every other page is then blank and scores
exactly 1.0, so you can read the Kade score straight off the leaderboard as **3 × score − 2**.
Either way, upload `submission.csv` to Kaggle and write the score in the results table.

In [ ]:
pred = os.path.join(out, "kade_test_predictions.csv")
base_csv = "sample_submission.csv"   # <-- or the team's latest full submission
if os.path.exists(pred):
    kade_report.merge_into_submission(base_csv, pred, os.path.join(run_dir, "submission.csv"))
else:
    print("This run didn't read the test pages. Only `baseline` and `final` do.")

## 8. Save the adapter to S3 (optional), then stop the instance

The run folder survives a Stop, but not a Delete, and teammates can't see it. Copy it to your
bucket to keep it. The adapter is about 130 MB; checkpoints are left out.

In [ ]:
bucket_name = "sinkorswim-doejohn"   # <-- EDIT ME: your own bucket
!aws s3 sync {run_dir} s3://{bucket_name}/badger-scribe/kade-lora/runs/{run_name}/ --exclude "checkpoints/*"

**Now stop the instance:** SageMaker → Notebook instances → select it → Actions → Stop.
Everything under `/home/ec2-user/SageMaker` (this repo, `kade-lora/`) is still there when you
start it again; re-run step 1 after each start.